In [17]:
import pandas as pd
import networkx as nx

def build_network_by_index(
    B_nodes_csv,
    AA_edges_csv,
    AB_edges_csv,
    has_header=True
):
    header = 0 if has_header else None
    G = nx.Graph()

    # Add Nodes
    nodes = pd.read_csv(B_nodes_csv, header=header)
    for i, row in nodes.iterrows():
        G.add_node(row.iloc[0], direction=row.iloc[1])

    # Add Edges
    for edge_csv in [AA_edges_csv, AB_edges_csv]:
        edges = pd.read_csv(edge_csv, header=header)
        for i, row in edges.iterrows():
            G.add_edge(row.iloc[0],row.iloc[1],direction=row.iloc[2])

    # Define layer 0
    layer_0 = []
    for n in nodes.iloc[:, 0]:
        if G.degree(n) > 0:
            layer_0.append(n)
    
    return G,layer_0


def build_layers(G, layer_0, max_depth=None):
    # list of sets --> layers[d] will contain all nodes at distance d (starting will change later)
    layers = []

    
    layers.append(set(layer_0))    # "layer 0" is the source nodes
    visited = set(layer_0)    # "visited" is all nodes we have already visited
    current_layer = set(layer_0)    # nodes at the current distance from layer 0
    depth = 0    # current distance from layer 0

    
    while current_layer: #until no new nodes are found
        if max_depth is not None and depth >= max_depth:    # until max depth (DO NOT REMOVE KEPT FOR DEBUGGING)
            break

        next_layer = set()  # next_layer nodes = distance (depth + 1)

        for node in current_layer:    # for every node in the current layer
            for neighbor in G.neighbors(node):     # look at all neighbors of this node
                if neighbor not in visited:     # add neighbor only if it hasn't appeared in any earlier layer
                    next_layer.add(neighbor)

        if not next_layer:     # if no new nodes were found, we are done
            break

        layers.append(next_layer)  # Save next layer

        visited.update(next_layer)  # Updated all visited layers (already discovered and assigned a layer)

        current_layer = next_layer  # Current layer is the next layer
        depth = depth + 1  # Increment depth

    return layers  # return list of layers: layers[d]

# For each node in layer 1(d) count all edges between layer 1(d) and layer 0(d-1) --> (where d represents layer)
# For each node in layer 1(d), create two variables called pos_votes and neg_votes. 

# deletes edges based on previous layer
"""
def filter_level(G, curr_layer, prev_layer, first,d):
    temp = 0 # DELETE LATER-----------------------
    for v in list(curr_layer):  # for every node v in the current layer
        
        pos_votes = 0
        neg_votes = 0
        pos_edges = []   # edges that voted positive
        neg_edges = []   # edges that voted negative

        for u in list(G.neighbors(v)):  # u is neighbors of v from previous layer
            if u in prev_layer:
                #if "direction" not in u and first==False:
                    #print("We will run into an error at",u)
                node_dir = float(G.nodes[u]["direction"])
                edge_dir = float(G[u][v]["direction"])
                vote = node_dir * edge_dir

                if vote > 0:
                    pos_votes += 1
                    log_file.write(f"{u} votes + \n")
                    pos_edges.append((u, v))
                else:
                    neg_votes += 1
                    log_file.write(f"{u} votes - \n")
                    neg_edges.append((u, v))

        log_file.write(f"NODE- {v},\nLAYER-{d},\nPOS_VOTES- {pos_votes},\nNEG_VOTES- {neg_votes}\n")

        if (v=="L-Lysine"):
            #print("Second TIME _-----------------------")
            print("Positive votes:",pos_votes)
            print("Negative votes:",neg_votes)
            print("Neighbors",list(G.neighbors(v)))
            #print("Second TIME _-----------------------")
        # if no votes, skip (otherwise dividing by zero)
        total = pos_votes + neg_votes
        if total == 0:
            print("ERROR")
            continue

        # set direction of v based on majority vote
        if pos_votes > neg_votes:
            G.nodes[v]["direction"] = 1
            minority_edges = neg_edges
        elif pos_votes < neg_votes:
            G.nodes[v]["direction"] = -1
            minority_edges = pos_edges
        else:
            minority_edges = pos_edges + neg_edges

        # calculate puc_value --> minority / total
        G.nodes[v]["puc_value"] = len(minority_edges) / total

        if G.nodes[v]["puc_value"] > 0.2:
            # delete all edges between v and nodes in the previous layer
            if first==True: #if layer 1, remove edge
                
                if ("direction" in G.nodes[v]):
                    del G.nodes[v]["direction"]
                for u in list(G.neighbors(v)):
                    if u in prev_layer:
                        G.remove_edge(u, v)
                        log_file.write(f"{v}-{u} edge is removed\n")
                        #print(first)
                log_file.write(f"{list(G.neighbors(v))} are neighbors of {v}\n")
            else: #all other layers, remove node
                if G.has_node(v):
                    G.remove_node(v)
                    log_file.write(f"{v} node is removed\n")
        else:
            # delete only minority-vote edges
            for a, b in minority_edges:
                G.remove_edge(a, b)
                log_file.write(f"{a} {b} edge is removed MINORITY CASE\n")
        if (v=="Dopamine"):
            print("Second TIME _-----------------------")
        log_file.write("\n\n")
    return G
"""
def filter_level(G, curr_layer, prev_layer, first, d, node_report):
    for v in list(curr_layer):

        pos_votes = 0
        neg_votes = 0
        pos_edges = []
        neg_edges = []

        for u in list(G.neighbors(v)):
            if u in prev_layer:
                node_dir = float(G.nodes[u]["direction"])
                edge_dir = float(G[u][v]["direction"])
                vote = node_dir * edge_dir

                if vote > 0:
                    pos_votes += 1
                    log_file.write(f"{u} votes + \n")
                    pos_edges.append((u, v))
                else:
                    neg_votes += 1
                    log_file.write(f"{u} votes - \n")
                    neg_edges.append((u, v))

        log_file.write(f"NODE- {v},\nLAYER-{d},\nPOS_VOTES- {pos_votes},\nNEG_VOTES- {neg_votes}\n")

        total = pos_votes + neg_votes

        # initialize this node's report entry (created/updated each time it's processed)
        report = node_report.setdefault(v, {
            "num_votes": 0,
            "up_votes": 0,
            "down_votes": 0,
            "frustration": None,
            "removed_node_majority": False,
            "removed_edges_layer1": False,
            "removed_minority_edges": False,
            "removed_same_level_edge": False,
        })
        report["num_votes"] = total
        report["up_votes"] = pos_votes
        report["down_votes"] = neg_votes

        if total == 0:
            print("ERROR")
            continue

        if pos_votes > neg_votes:
            G.nodes[v]["direction"] = 1
            minority_edges = neg_edges
        elif pos_votes < neg_votes:
            G.nodes[v]["direction"] = -1
            minority_edges = pos_edges
        else:
            minority_edges = pos_edges + neg_edges

        puc_value = len(minority_edges) / total
        G.nodes[v]["puc_value"] = puc_value
        report["frustration"] = puc_value

        if puc_value > 0.2:
            if first == True:
                if "direction" in G.nodes[v]:
                    del G.nodes[v]["direction"]
                for u in list(G.neighbors(v)):
                    if u in prev_layer:
                        G.remove_edge(u, v)
                        log_file.write(f"{v}-{u} edge is removed\n")
                report["removed_edges_layer1"] = True
                log_file.write(f"{list(G.neighbors(v))} are neighbors of {v}\n")
            else:
                if G.has_node(v):
                    G.remove_node(v)
                    log_file.write(f"{v} node is removed\n")
                report["removed_node_majority"] = True
        else:
            for a, b in minority_edges:
                G.remove_edge(a, b)
                log_file.write(f"{a} {b} edge is removed MINORITY CASE\n")
            if minority_edges:
                report["removed_minority_edges"] = True

        log_file.write("\n\n")
    return G
"""
# deletes edges in same level
def same_level_edges(G, curr_layer,d):
    curr_layer = set(curr_layer)  

    for u in list(curr_layer): # for u in currrent layer
        if not G.has_node(u):
            continue
            
        for v in list(G.neighbors(u)): # for v in current layer that is a neighbor of u
            if v in curr_layer and G.has_node(v):
                if "direction" not in G.nodes[u] or "direction" not in G.nodes[v]:
                    #print(u,"is given another chance")
                    continue
                    
                if (v=="L-Glutamine" or u=="L-Glutamine"):
                    print("Not being removed",u,G.nodes[u]["direction"])
                    print("Not being removed",v,G.nodes[v]["direction"])
                
                node_prod = float(G.nodes[u]["direction"]) * float(G.nodes[v]["direction"])
                edge_dir = float(G[u][v]["direction"])

                if node_prod != edge_dir:  # if product of node directions != edge direction, delete edge
                    if G.has_edge(u, v):  # avoid double-delete since undirected
                        log_file.write(f"SAME EDGE REMOVAL {v}-{u} edge is removed in layer{d}\n\n\n")
                        G.remove_edge(u, v)

    return G
"""
def same_level_edges(G, curr_layer, d, node_report):
    curr_layer = set(curr_layer)

    for u in list(curr_layer):
        if not G.has_node(u):
            continue

        for v in list(G.neighbors(u)):
            if v in curr_layer and G.has_node(v):
                if "direction" not in G.nodes[u] or "direction" not in G.nodes[v]:
                    continue

                node_prod = float(G.nodes[u]["direction"]) * float(G.nodes[v]["direction"])
                edge_dir = float(G[u][v]["direction"])

                if node_prod != edge_dir:
                    if G.has_edge(u, v):
                        log_file.write(f"SAME EDGE REMOVAL {v}-{u} edge is removed in layer{d}\n\n\n")
                        G.remove_edge(u, v)
                        for n in (u, v):
                            node_report.setdefault(n, {
                                "num_votes": 0, "up_votes": 0, "down_votes": 0,
                                "frustration": None, "removed_node_majority": False,
                                "removed_edges_layer1": False, "removed_minority_edges": False,
                                "removed_same_level_edge": False,
                            })["removed_same_level_edge"] = True

    return G

In [19]:
import pandas as pd

log_file = open("log_file.txt","w")

#puc_delete_threshold=0.2
graph_A, layer_0 = build_network_by_index(
    "0. Original Files/cpx_nodes.csv",
    "0. Original Files/csf_csf.csv",
    "0. Original Files/alt_csf_cpx.csv",
    has_header=True
)
print("Starting",len(graph_A.nodes())-len(layer_0))

layers = build_layers(graph_A, layer_0, max_depth=None)
print("Layer 0", len(layer_0))
print(graph_A)

# ADDED ------------------------------------------------------------------------------------------------------------
original_nodes = list(graph_A.nodes())   # full original node set, before any removals
node_report = {}
# ADDED ------------------------------------------------------------------------------------------------------------

d = 1 #Start from distance = 1
while d < len(layers):
    
    # DEBUG -----------------------------
    print("Total layers:", len(layers))
    # Number of nodes per layer
    sum = 0
    for i, layer in enumerate(layers):
        if i>0:
            sum = sum + len(layer)
            print(f"Layer {i}: {len(layer)} nodes")
            #if (i>0):
                #print(f"Layer {i}: {layer} nodes")
    print("Total number of nodes kept",sum)
    # -----------------------------------

    print("Starting with:",d)
    prev_layer = set(layers[d - 1])   # define previous layer
    curr_layer = layers[d]   # define current layer
    
    if (d == -1):
        #graph_A = filter_level(graph_A, curr_layer, prev_layer, True,d)
        graph_A = filter_level(graph_A, curr_layer, prev_layer, True, d, node_report)

        print("LAYER NOT FOUND")
    else:
        #graph_A = filter_level(graph_A, curr_layer, prev_layer, False,d)
        graph_A = filter_level(graph_A, curr_layer, prev_layer, False, d, node_report)

        
    #graph_A = same_level_edges(graph_A, curr_layer,d)
    graph_A = same_level_edges(graph_A, curr_layer, d, node_report)

    
    layers = build_layers(graph_A, layer_0, max_depth=None)
    print("done with layer:",d,"\n")
    d += 1



rows = []
for d in range(1, len(layers)):
    for node in layers[d]:
        rows.append({
            "node": node,
            "direction": graph_A.nodes[node].get("direction"),
            "puc_value": graph_A.nodes[node].get("puc_value")
        })
    

new_rows = []
for node in graph_A.nodes():
    new_rows.append({
        "node": node,
        "direction": graph_A.nodes[node].get("direction"),
        "puc_value": graph_A.nodes[node].get("puc_value")
    })
print("ROWS",len(rows))
new_df = pd.DataFrame(rows, columns=["node", "direction", "puc_value"])
new_df.to_csv("alt_new.csv")

df = pd.DataFrame(rows, columns=["node", "direction", "puc_value"])
df.to_csv("alt_CSF Effect Sign Predictions.csv", index=False)


"""
import pandas as pd
import networkx as nx

# node_to_layer = final layers after filtering
node_to_layer = {}
for li, layer_nodes in enumerate(layers):
    for n in layer_nodes:
        node_to_layer[n] = li

# Find all nodes reachable from layer_0
reachable_from_layer0 = set()
for src in layer_0:
    if src in graph_A:
        reachable_from_layer0.update(nx.node_connected_component(graph_A, src))

# Edge pairs starting from layer 1 (same layer or adjacent layers)
edge_rows = []
for u, v in graph_A.edges():
    lu = node_to_layer.get(u)
    lv = node_to_layer.get(v)

    # Skip edges if either node is not reachable from layer_0
    if u not in reachable_from_layer0 or v not in reachable_from_layer0:
        continue

    # Skip edges involving layer_0 nodes themselves
    if lu == 0 or lv == 0:
        continue

    # Keep only same-layer or adjacent-layer edges
    if abs(lu - lv) > 1:
        continue
        
    edge_rows.append({"node1": u, "node2": v})

df_edges = pd.DataFrame(edge_rows, columns=["node1", "node2"])
df_edges.to_csv("new_csf-csf_node_pairs.csv", index=False)
print("done")
print("edges written:", len(edge_rows))
"""


import pandas as pd

# node_to_layer = final layers after filtering
node_to_layer = {}
for li, layer_nodes in enumerate(layers):
    for n in layer_nodes:
        node_to_layer[n] = li

# Edge pairs starting from layer 1 (same layer or adjacent layers)
edge_rows = []
for u, v in graph_A.edges():
    lu = node_to_layer.get(u)
    lv = node_to_layer.get(v)

    # Start from layer 1 only (exclude minus layer 0)
    #if not (lu == 0 and lv == 1):
    if (lu is None or lv is None):
        print(lu)
        print(lv)
        print("ERROR")
        continue

    if lu + lv != 1:
        continue
    
    edge_rows.append({"node1": u,"node2": v})

df_edges = pd.DataFrame(edge_rows, columns=["node1", "node2"])
df_edges.to_csv("alt_new_csf-cpx_node_pairs.csv", index=False)
print("done")
print("edges written:", len(edge_rows))


# layer 0 gets layer = 0; everything else looks up node_to_layer; missing => None
node_to_layer = {}
for li, layer_nodes in enumerate(layers):
    for n in layer_nodes:
        node_to_layer[n] = li

full_report_rows = []
for node in original_nodes:
    r = node_report.get(node, {})
    full_report_rows.append({
        "node": node,
        "num_votes": r.get("num_votes"),
        "up_votes": r.get("up_votes"),
        "down_votes": r.get("down_votes"),
        "frustration": r.get("frustration"),
        "removed_node_majority": r.get("removed_node_majority", False),
        "removed_edges_layer1": r.get("removed_edges_layer1", False),
        "removed_minority_edges": r.get("removed_minority_edges", False),
        "removed_same_level_edge": r.get("removed_same_level_edge", False),
        "layer": node_to_layer.get(node)  # None if not in any final layer
    })

full_report_df = pd.DataFrame(full_report_rows, columns=[
    "node", "num_votes", "up_votes", "down_votes", "frustration",
    "removed_node_majority", "removed_edges_layer1", "removed_minority_edges",
    "removed_same_level_edge", "layer"
])
full_report_df.to_csv("alt_full_node_report.csv", index=False)


Starting 897
Layer 0 118
Graph with 1015 nodes and 2087 edges
Total layers: 4
Layer 1: 88 nodes
Layer 2: 120 nodes
Layer 3: 4 nodes
Total number of nodes kept 212
Starting with: 1
done with layer: 1 

Total layers: 4
Layer 1: 73 nodes
Layer 2: 117 nodes
Layer 3: 7 nodes
Total number of nodes kept 197
Starting with: 2
done with layer: 2 

Total layers: 4
Layer 1: 73 nodes
Layer 2: 59 nodes
Layer 3: 7 nodes
Total number of nodes kept 139
Starting with: 3
done with layer: 3 

ROWS 139
done
edges written: 122
